In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


In [ ]:
from pyspark.sql import Row

data = [
    # ✅ valid row
    Row(song_id=1, artist_id=10, artist_name="Artist A", song_title="Song X", duration=210, year=2020),

    # ❌ duplicate PK (song_id = 1)
    Row(song_id=1, artist_id=10, artist_name="Artist A", song_title="Song X", duration=210, year=2020),

    # ❌ NULL artist_name
    Row(song_id=2, artist_id=11, artist_name=None, song_title="Song Y", duration=180, year=2019),

    # ❌ duration <= 0
    Row(song_id=3, artist_id=12, artist_name="Artist C", song_title="Song Z", duration=0, year=2018),

    # ❌ duplicate (artist_id, song_title)
    Row(song_id=4, artist_id=13, artist_name="Artist D", song_title="Song W", duration=200, year=2021),
    Row(song_id=5, artist_id=13, artist_name="Artist D", song_title="Song W", duration=200, year=2021),

    # ❌ NULL song_title
    Row(song_id=6, artist_id=14, artist_name="Artist E", song_title=None, duration=190, year=2022),
]

df_test = spark.createDataFrame(data)
df_test.show(truncate=False)
df_test.printSchema()

df_test.createOrReplaceTempView("songs_raw")


In [ ]:
df_test.where("NOT year between 2018 and 2019").show(5, 0)

In [ ]:
import dqp as dp

@dp.materialized_view(
    comment="Test songs dataset",
    fail_fast=False  # important so all checks run
)

@dp.expect_no_duplicates(
    name="check duplicates",
    columns=["artist_name", "song_title"],
    severity="WARN"
)
@dp.expect_primary_key(
    name="valid_primary_key",
    columns="song_id",
    severity="ERROR"
)
@dp.expect(name="valid_year", rule="year BETWEEN 2018 AND 2020", severity="ERROR")
@dp.expect(
    name="valid_duration",
    rule="duration > 0",
    severity="ERROR"
)
def songs_test():
    return spark.read.table("songs_raw")


In [ ]:
try:
    result_df = songs_test()
    result_df.show()
except Exception as e:
    print("Pipeline failed:", e)
